In [1]:
import torch, torch.nn as nn, timm, requests
from torchvision import transforms
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

dtype = torch.float16 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen = AutoModelForCausalLM.from_pretrained(model_name,
                                                  torch_dtype=dtype,
                                                  device_map="auto")
hidden_size = qwen.config.hidden_size # 4096 for 8-B

vit = timm.create_model("vit_large_patch16_224", pretrained=True)
vit.head = nn.Identity()
vit = vit.to(device).half().eval()               # cast weights to fp16

img = Image.open("cows.jpeg").convert("RGB")
prep = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=vit.default_cfg["mean"], std=vit.default_cfg["std"])
])
img_t = prep(img).unsqueeze(0).to(device).half() # fp16

with torch.no_grad():
    vit_feats = vit.forward_features(img_t).half()  # [B,P,1024] fp16

# --- visual mapper (cross-attention) ----------------------------------------
class VisualMapper(nn.Module):
    def __init__(self, n_tok=8, vdim=1024, hdim=hidden_size):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, n_tok, hdim, dtype=dtype, device=device))
        self.proj  = nn.Linear(vdim, hdim, bias=True).to(device).half()
        self.attn  = nn.MultiheadAttention(hdim, 8, batch_first=True).to(device).half()
    def forward(self, v):
        b = v.size(0)
        v = self.proj(v)               # [B,P,H]
        q = self.query.expand(b, -1, -1)
        return self.attn(q, v, v)[0]   # [B,n_tok,H]

mapper = VisualMapper().eval()
vis_tok = mapper(vit_feats)            # fp16

prompt = "What do you see in this image?"
chat_txt = tokenizer.apply_chat_template(
    [{"role":"user","content":prompt}],
    tokenize=False, add_generation_prompt=True
)
ids = tokenizer(chat_txt, return_tensors="pt").input_ids.to(device)
txt_emb = qwen.model.embed_tokens(ids)          # already fp16


inp_emb  = torch.cat([vis_tok, txt_emb], dim=1)  # [1, n_tok+T, H] fp16
attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)

with torch.no_grad():
    out = qwen.generate(inputs_embeds=inp_emb,
                        attention_mask=attn_msk,
                        max_new_tokens=128,
                        do_sample=False)

gen_ids  = out[0][ids.shape[1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-30 17:49:20.842227: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-30 17:49:20.855899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748594960.873090 2799794 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748594960.876848 2799794 cuda_blas

ImportError: cannot import name 'builder' from 'google.protobuf.internal' (/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/google/protobuf/internal/__init__.py)

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForCausalLM, AutoTokenizer, CLIPVisionModel, CLIPImageProcessor
from huggingface_hub import hf_hub_download

dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Qwen language model
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
hidden_size = qwen.config.hidden_size

# Load CLIP vision model
clip_model_name = "openai/clip-vit-large-patch14"
clip = CLIPVisionModel.from_pretrained(clip_model_name).to(device).half().eval()
image_processor = CLIPImageProcessor.from_pretrained(clip_model_name)

# Load LLaVA projector weights
projector_path = hf_hub_download("liuhaotian/llava-v1.5-7b", "mm_projector.bin")
projector_state = torch.load(projector_path, map_location="cpu")
proj = nn.Sequential(
    nn.Linear(1024, hidden_size),
    nn.GELU(),
    nn.Linear(hidden_size, hidden_size)
).to(device).half()
proj.load_state_dict({k.replace("mm_projector.", ""): v.half() for k, v in projector_state.items()})

# Load and preprocess image
img = Image.open("cows.jpeg").convert("RGB")
img_t = image_processor(images=img, return_tensors="pt")["pixel_values"].to(device).half()

# Extract visual features
with torch.no_grad():
    vision_outputs = clip(img_t, output_hidden_states=True)
    image_features = vision_outputs.hidden_states[-2][:, 1:]  # Exclude CLS token
    vis_tok = proj(image_features)

# Prepare prompt
prompt = "What do you see in this image?"
chat_txt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True
)
ids = tokenizer(chat_txt, return_tensors="pt").input_ids.to(device)
txt_emb = qwen.model.embed_tokens(ids)

# Concatenate visual tokens and text embeddings
inp_emb = torch.cat([vis_tok, txt_emb], dim=1)
attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)

# Generate response
with torch.no_grad():
    out = qwen.generate(
        inputs_embeds=inp_emb,
        attention_mask=attn_msk,
        max_new_tokens=128,
        do_sample=False
    )

gen_ids = out[0][ids.shape[1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))


In [ ]:
import torch, torch.nn as nn, timm, requests
from torchvision import transforms
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
hidden_size = qwen.config.hidden_size

vit = timm.create_model("vit_large_patch16_224", pretrained=True)
vit.head = nn.Identity()
vit = vit.to(device).half().eval()

img = Image.open("cows.jpeg").convert("RGB")
prep = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=vit.default_cfg["mean"], std=vit.default_cfg["std"])
])
img_t = prep(img).unsqueeze(0).to(device).half()

with torch.no_grad():
    vit_feats = vit.forward_features(img_t).half()

class VisualMapper(nn.Module):
    def __init__(self, n_tok=8, vdim=1024, hdim=hidden_size):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, n_tok, hdim, dtype=dtype, device=device))
        self.proj  = nn.Linear(vdim, hdim, bias=True).to(device).half()
        self.attn  = nn.MultiheadAttention(hdim, 8, batch_first=True).to(device).half()
    def forward(self, v):
        b = v.size(0)
        v = self.proj(v)
        q = self.query.expand(b, -1, -1)
        return self.attn(q, v, v)[0]

mapper = VisualMapper().to(device).half()
opt = torch.optim.AdamW(mapper.parameters(), lr=5e-4)

caption = "A herd of cows grazing on grass under a blue sky." + tokenizer.eos_token
cap_ids = tokenizer(caption, return_tensors="pt").input_ids.to(device)

opt.zero_grad()
vis_tok = mapper(vit_feats)
cap_emb = qwen.model.embed_tokens(cap_ids[:, :-1])
inp_emb = torch.cat([vis_tok, cap_emb], 1)
attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)

labels = torch.full(inp_emb.shape[:2], -100, dtype=torch.long, device=device)
labels[:, vis_tok.shape[1]:] = cap_ids[:, 1:]

loss = qwen(inputs_embeds=inp_emb, attention_mask=attn_msk, labels=labels).loss
loss.backward()
opt.step()

mapper.eval()
vis_tok = mapper(vit_feats)
prompt = "What do you see in this image?"
chat_txt = tokenizer.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
ids = tokenizer(chat_txt, return_tensors="pt").input_ids.to(device)
txt_emb = qwen.model.embed_tokens(ids)

inp_emb = torch.cat([vis_tok, txt_emb], 1)
attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)

with torch.no_grad():
    out = qwen.generate(inputs_embeds=inp_emb, attention_mask=attn_msk, max_new_tokens=128, do_sample=False)

gen_ids = out[0][ids.shape[1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))


In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForCausalLM, AutoTokenizer, CLIPVisionModel, CLIPImageProcessor
from huggingface_hub import hf_hub_download

dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Qwen language model
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
hidden_size = qwen.config.hidden_size

# Load CLIP vision model
clip_model_name = "openai/clip-vit-large-patch14"
clip = CLIPVisionModel.from_pretrained(clip_model_name).to(device).half().eval()
image_processor = CLIPImageProcessor.from_pretrained(clip_model_name)

# Define projector with same structure as LLaVA's mm_projector
proj = nn.Sequential(
    nn.Linear(1024, hidden_size),
    nn.GELU(),
    nn.Linear(hidden_size, hidden_size)
).to(device)

# Download and load projector weights
projector_path = hf_hub_download("liuhaotian/llava-v1.5-7b", "mm_projector.bin")
projector_state = torch.load(projector_path, map_location="cpu")

# Rename keys to match Sequential model
new_state_dict = {}
for k, v in projector_state.items():
    k = k.replace("model.mm_projector.", "")  # Remove prefix
    new_state_dict[k] = v.to(device)

proj.load_state_dict(new_state_dict)

# Load and preprocess image
img = Image.open("cows.jpeg").convert("RGB")
img_t = image_processor(images=img, return_tensors="pt")["pixel_values"].to(device).half()

# Extract visual features using CLIP
with torch.no_grad():
    vision_outputs = clip(img_t, output_hidden_states=True)
    image_features = vision_outputs.hidden_states[-2][:, 1:]  # Skip CLS token
    vis_tokens = proj(image_features)

# Prepare prompt
prompt = "<image>What do you see in this image?"
chat_txt = tokenizer.apply_chat_template(
    [{"role": "user", "content": f"<image>{prompt}"}],
    tokenize=False,
    add_generation_prompt=True
)
input_ids = tokenizer(chat_txt, return_tensors="pt").input_ids.to(device)
txt_embeddings = qwen.model.embed_tokens(input_ids)

# Concatenate visual tokens and text embeddings
inputs_embeds = torch.cat([vis_tokens, txt_embeddings], dim=1)
attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

# Generate response
with torch.no_grad():
    outputs = qwen.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False
    )

# Decode and print the generated response
generated_ids = outputs[0][input_ids.shape[1]:]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from vit_pytorch.simple_vit_with_patch_dropout import SimpleViT
from vit_pytorch.extractor import Extractor
from coca_pytorch.coca_pytorch import CoCa

# Set device and data type
dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Qwen3-8B model and tokenizer
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
hidden_size = qwen.config.hidden_size

# Define the Vision Transformer
vit = SimpleViT(
    image_size=256,
    patch_size=32,
    num_classes=1000,
    dim=1024,
    depth=6,
    heads=16,
    mlp_dim=2048,
    patch_dropout=0.5
)
vit = Extractor(vit, return_embeddings_only=True, detach=False)

# Initialize CoCa
coca = CoCa(
    dim=512,
    img_encoder=vit,
    image_dim=1024,
    num_tokens=4000,
    unimodal_depth=6,
    multimodal_depth=6,
    dim_head=64,
    heads=8,
    caption_loss_weight=1.0,
    contrastive_loss_weight=1.0,
).to(device).half().eval()

# Define projection layer
proj = nn.Linear(512, hidden_size, bias=False).to(device).half()

# Define image preprocessing
prep = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# Load dataset
ds = load_dataset("zmao/food_img_caption_small", split="train")

# Define collate function
def collate_fn(batch):
    imgs = torch.stack([prep(x["image"]) for x in batch]).to(device).half()
    caps = [x["text"] + tokenizer.eos_token for x in batch]
    cap_ids = tokenizer(caps, return_tensors="pt", padding=True).input_ids.to(device)
    return imgs, cap_ids

# Create DataLoader
loader = torch.utils.data.DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

# Define optimizer
opt = torch.optim.AdamW(list(coca.parameters()) + list(proj.parameters()), lr=1e-4)

# Training loop
for epoch in range(1):
    print(f"Epoch {epoch+1}")
    coca.train()
    for imgs, cap_ids in loader:
        # Extract image embeddings
        with torch.no_grad():
            _, img_emb = coca(
                text=torch.randint(0, 1, (imgs.size(0), 1), device=device),
                images=imgs,
                return_embeddings=True
            )
        vis_tok = proj(img_emb).unsqueeze(1)  # [B, 1, hidden_size]

        # Get text embeddings
        cap_emb = qwen.model.embed_tokens(cap_ids[:, :-1])

        # Concatenate image and text embeddings
        inp_emb = torch.cat([vis_tok, cap_emb], dim=1)
        attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)

        # Prepare labels
        labels = torch.full(inp_emb.shape[:2], -100, dtype=torch.long, device=device)
        labels[:, vis_tok.size(1):] = cap_ids[:, 1:]

        # Compute loss
        loss = qwen(inputs_embeds=inp_emb, attention_mask=attn_msk, labels=labels).loss
        loss.backward()
        print(f"Loss: {loss.item():.4f}")
        opt.step()
        opt.zero_grad()

# Evaluation
coca.eval()
first = ds[0]
img = first["image"]
img_t = prep(img).unsqueeze(0).to(device).half()
with torch.no_grad():
    _, img_emb = coca(
        text=torch.randint(0, 1, (1, 1), device=device),
        images=img_t,
        return_embeddings=True
    )
    vis_tok = proj(img_emb).unsqueeze(1)  # [1, 1, hidden_size]

    prompt = "What do you see in this image?"
    chat_txt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )
    ids = tokenizer(chat_txt, return_tensors="pt").input_ids.to(device)
    txt_emb = qwen.model.embed_tokens(ids)
    inp_emb = torch.cat([vis_tok, txt_emb], dim=1)
    attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)
    out = qwen.generate(inputs_embeds=inp_emb, attention_mask=attn_msk, max_new_tokens=128, do_sample=False)
    gen = out[0][ids.shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=True))
